# B-spline fitting to scattered data

Adapted from the [B-spline section of the ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621#b-spline-fitting).
Explore parametric curves, a trefoil surface, and image reconstruction.
Random samples are seeded; the default examples require no downloads.

**API scope:** ANTsTorch currently supports cubic fits with 2-D or 3-D
parametric domains. The tutorial's 1-D curve fits therefore use **ANTsPy**
explicitly below. Surface and image fits use **ANTsTorch** and return tensors
with layout `(batch, components, reversed spatial axes)`.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

## Curves: ANTsPy reference examples
A curve has one parameter even when its points lie in two dimensions.
Fit an open noisy Gaussian curve and a closed hypocycloid. These are reference
calls to `ants`, not demonstrations of 1-D fitting in ANTsTorch.

In [ ]:
rng = np.random.default_rng(42)
count = 100
coordinate = np.linspace(-4, 4, count)
open_points = np.stack([coordinate, np.exp(-coordinate**2) + rng.uniform(-0.1, 0.1, count)], axis=1)
phase = np.linspace(0, 2*np.pi, count, endpoint=False)
closed_points = np.stack([9*np.cos(phase) + 3*np.cos(3*phase),
                          9*np.sin(phase) - 3*np.sin(3*phase)], axis=1)
closed_points += rng.uniform(-1, 1, closed_points.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, points, closed in zip(axes, [open_points, closed_points], [False, True]):
    parameter = np.linspace(0, 1, count, endpoint=not closed)[:, None]
    curve = np.asarray(ants.fit_bspline_object_to_scattered_data(
        points, parameter, parametric_domain_origin=[0.],
        parametric_domain_spacing=[1/(count-1)], parametric_domain_size=[count],
        is_parametric_dimension_closed=[closed],
        number_of_fitting_levels=3, mesh_size=4 if closed else 1,
    ))
    assert curve.shape == (count, 2) and np.isfinite(curve).all()
    ax.scatter(*points.T, s=10, label="Samples")
    ax.plot(*curve.T, label="ANTsPy fit")
    ax.set_title("Closed curve" if closed else "Open curve")
    ax.legend(); ax.grid(True)
plt.show()

## Surface: ANTsTorch fit with three output components
Two parameters describe this surface embedded in 3-D. Fit its three coordinates
together in one tensor call. A smaller sampling grid than the tutorial keeps
this example quick. Both parameter axes are periodic.

In [ ]:
from antstorch.bspline_flows import fit_bspline_object_to_scattered_data

sample_size = 32
u, v = np.meshgrid(np.arange(sample_size)/sample_size,
                   np.arange(sample_size)/sample_size, indexing="xy")
angle, tube = 4*np.pi*u - 2*np.pi, 2*np.pi*v - np.pi
radius = np.cos(tube) + 4.5 + 1.5*np.sin(1.5*angle)
points = np.stack([radius*np.cos(angle), radius*np.sin(angle),
                   np.sin(tube) + 2*np.cos(1.5*angle)], axis=-1)
parameters = np.stack([u.ravel(), v.ravel()], axis=1)
surface = fit_bspline_object_to_scattered_data(
    points.reshape(-1, 3), parameters,
    parametric_domain_origin=(0., 0.), parametric_domain_spacing=(1/63, 1/63),
    parametric_domain_size=(64, 64), is_parametric_dimension_closed=(True, True),
    number_of_fitting_levels=3, mesh_size=(4, 7), device=device, dtype=torch.float64,
)
assert surface.shape == (1, 3, 64, 64) and torch.isfinite(surface).all()
coordinates = surface[0].detach().cpu().numpy()
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(*coordinates, cmap="viridis", linewidth=0)
ax.set_title("ANTsTorch fitted trefoil surface")
plt.show()

## Image: reconstruct from scattered intensity samples
The tutorial samples r16. Here a known smooth phantom makes the default
workflow offline and allows an error check. Set `use_sample_r16=True` to
download and fit r16 instead. Image samples have one output component and
two spatial parameters; ANTsTorch reverses the spatial axes in its output.

In [ ]:
use_sample_r16 = False  # True downloads an ANTsPy sample on first use.
if use_sample_r16:
    reference = ants.image_read(ants.get_ants_data("r16")).clone("float")
    reference = ants.resample_image(reference, (64, 64), use_voxels=True)
else:
    grid_x, grid_y = np.meshgrid(np.linspace(-1, 1, 48), np.linspace(-1, 1, 40), indexing="ij")
    phantom = (np.exp(-4*(grid_x**2 + grid_y**2)) + 0.2*grid_x).astype("float32")
    reference = ants.from_numpy(phantom)
array = reference.numpy()
sample_x = rng.integers(0, array.shape[0], 2000)
sample_y = rng.integers(0, array.shape[1], 2000)
locations = np.stack([sample_x, sample_y], axis=1)
samples = array[sample_x, sample_y, None]
fitted = fit_bspline_object_to_scattered_data(
    samples, locations, parametric_domain_origin=(0., 0.),
    parametric_domain_spacing=(1., 1.), parametric_domain_size=array.shape,
    number_of_fitting_levels=4, mesh_size=1, device=device, dtype=torch.float64,
)
reconstruction = fitted[0, 0].detach().cpu().numpy().T
assert reconstruction.shape == array.shape and np.isfinite(reconstruction).all()
rmse = float(np.sqrt(np.mean((array - reconstruction)**2)))
baseline = float(np.sqrt(np.mean((array - array.mean())**2)))
assert rmse < baseline
print(f"Image RMSE: {rmse:.4f}; constant-image baseline: {baseline:.4f}")
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, values, title in zip(axes, [array, reconstruction, array-reconstruction],
                            ["Reference", "ANTsTorch fit", "Residual"]):
    ax.imshow(values.T, origin="lower", cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()

Increasing fitting levels or mesh size permits finer detail and can also fit
noise. Compare held-out samples when choosing these parameters for real data.
`locations` above are voxel indices in a unit-spaced fitting domain, independent
of the input image's physical metadata; output arrays must be given the original
spacing, origin and direction if converted back into ANTs images.